In [1]:
# init
from pathlib import Path
from importlib import reload
import toolsGeneral.main as tgm
import toolsGeneral.logger as tgl
import toolsPandas.helpers as tph
import toolsOSM.overpass as too
import toolsSync.main as tsm
import os
import boto3
from dotenv import load_dotenv
import pandas as pd
import copy
import re
import shutil

def pckgs_reload():
    reload(tgm)
    reload(tph)
    reload(tsm)
    reload(too)

ROOT = Path.cwd().parents[0]
DATA_DIR = ROOT / "data"
CLEANED_DIR = ROOT / "data" / "cleaned"
RAW_DIR = DATA_DIR / 'raw/osm countries queries'
logger = tgl.initiate_logger('logger')

In [2]:
# init
# state files
process_state_file = Path(DATA_DIR / 'process_state.json')
process_state = tgm.load(process_state_file)
print(len(process_state))
dups_test_state_file = Path(DATA_DIR / 'dups_test_state.json')
dups_test_state = tgm.load(dups_test_state_file)
print(len(dups_test_state))
first_level_test_state_file = Path(DATA_DIR / 'first_level_test_state.json')
first_level_test_state = tgm.load(first_level_test_state_file)
print(len(first_level_test_state))

214
214
70


In [3]:
# init
load_dotenv()
bucket_name = os.environ["B2_BUCKET_NAME"]
session = boto3.session.Session()
s3 = session.client(
    service_name="s3",
    aws_access_key_id=os.environ["B2_KEY_ID"],
    aws_secret_access_key=os.environ["B2_APPLICATION_KEY"],
    endpoint_url=os.environ["B2_ENDPOINT"]
)

config = {'root':ROOT, 's3':s3, 'logger':logger}

sovereign_countries = tgm.load(DATA_DIR / 'sovereign countries.json')
print(len(sovereign_countries))

214


# read bucket

In [4]:
task_meta = {
    'scrape':{'dir':Path("data/raw/osm countries queries/")}, 
    'clean':{'dir':Path("data/cleaned/")}, 
    'test_basic':{'dir':Path("data/tests results/osm basic test")},
    'test_first_level':{'dir':Path("data/tests results/osm first level test")},
    'test_duplicates':{'dir':Path("data/tests results/osm duplicates test")}
}
# task_responses = {t:s3.list_objects_v2(Bucket=bucket_name, Prefix=task_meta[t]['dir'].as_posix()) for t in task_meta.keys()}
task_contents = {t:tsm.get_bucket_contents(s3, bucket_name, task_meta[t]['dir'].as_posix()) for t in task_meta.keys()}
print([len(res) for task, res in task_contents.items()])

sync_state = copy.deepcopy(task_meta)
for task, res in task_contents.items():
    sync_state[task]['b2_countries'] = {Path(obj['Key']).parent.name for obj in res if Path(obj['Key']).parent.name in sovereign_countries}
    sync_state[task]['local_countries'] = {dir.name for dir in (ROOT / sync_state[task]['dir']).glob('*') if dir.name in sovereign_countries}
    sync_state[task]['b2_files'] = {obj['Key'] for obj in res}
    sync_state[task]['local_files'] = {dir.relative_to(ROOT).as_posix() for dir in (ROOT / sync_state[task]['dir']).rglob('*') if dir.is_file()}
    b2_countries_not_in_local = tgm.complement(
        sync_state[task]['b2_countries'],
        sync_state[task]['local_countries']
    )
    b2_files_not_in_local = tgm.complement(sync_state[task]['b2_files'], sync_state[task]['local_files'])
    sync_state[task]['b2_files_not_in_local'] = b2_files_not_in_local
    local_files_not_in_b2 = tgm.complement(sync_state[task]['local_files'], sync_state[task]['b2_files'])
    sync_state[task]['local_files_not_in_b2'] = local_files_not_in_b2
    
# get len
sync_state_resume = copy.deepcopy(sync_state)
for task,val in sync_state_resume.items():
    for k,v in sync_state_resume[task].items():
        if k in ['dir']: continue
        sync_state_resume[task][k] = len(v)

[1181, 216, 214, 109, 56]


In [5]:
pd.DataFrame(sync_state_resume).T

,dir,b2_countries,local_countries,b2_files,local_files,b2_files_not_in_local,local_files_not_in_b2
scrape,data\raw\osm countries queries,214,95,1181,367,814,0
clean,data\cleaned,214,95,216,97,119,0
test_basic,data\tests results\osm basic test,214,84,214,85,130,1
test_first_level,data\tests results\osm first level test,69,66,109,76,33,0
test_duplicates,data\tests results\osm duplicates test,33,32,56,38,18,0


In [6]:
temp = sync_state['test_first_level']['b2_countries']
print(len(temp))

69


In [91]:
temp = tgm.complement(sync_state['test_first_level']['b2_countries'], sync_state['test_first_level']['local_countries'] )
print(len(temp))
temp

1


['UnitedStates']

In [57]:
print(len(tgm.intersection(sync_state['scrape']['b2_countries'], sync_state['scrape']['local_countries'] )))

83


# download state files

In [ ]:
# RUN 'python scripts/pull_from_B2.py' 

In [220]:
process_state_new_file = Path(DATA_DIR / "process_state_new.json")
process_state_new = tgm.load(process_state_new_file)

In [221]:
[process_state_new[c]['test_basic'] for c in countries_missing]

[{'status': 'ok', 'last_run': '2026-01-25T19:13:28', 'error': None},
 {'status': 'ok', 'last_run': '2026-01-25T19:13:28', 'error': None},
 {'status': 'ok', 'last_run': '2026-01-25T19:13:27', 'error': None},
 {'status': 'ok', 'last_run': '2026-01-25T19:13:28', 'error': None}]

### Replace after review

In [93]:
replace_files = [
    Path(DATA_DIR / "process_state.json"), 
    Path(DATA_DIR / "first_level_test_state.json"), 
    Path(DATA_DIR / "dups_test_state.json"),
    Path(CLEANED_DIR / 'dups_id.pkl'),
    Path(CLEANED_DIR / 'ids.pkl'),
]

In [94]:
for file in replace_files:
    if file.exists():
        file.unlink()
    new_file = file.with_name(file.stem + '_new' + file.suffix)
    new_file.rename(file)

### dups and ids from cleaned

In [82]:
ids_path = DATA_DIR / 'cleaned' / 'ids.pkl'
ids = tgm.load(ids_path)
print(len(ids))

dups_path = DATA_DIR / 'cleaned' / 'dups_id.pkl'
dups_id = tgm.load(dups_path)
print(len(dups_id))

223211
21071


In [65]:
tsm.download_file_from_bucket(bucket_name, ids_path.relative_to(ROOT), s3, ids_path, logger)

# download countries files

In [92]:
task = 'test_first_level'
to_download_countries = {Path(file).parent.name for file in sync_state[task]['b2_files_not_in_local']}
to_download_countries = ["Vietnam"]
print(len(to_download_countries))
print(to_download_countries)

1
['Vietnam']


In [74]:
# In oder to sync them more easily, just delete the local directory
for country in to_download_countries:
    try:
        dir = (ROOT / task_meta[task]['dir'] / country)
        dir.rmdir
        print(f"Directory deleted: {dir}")
    except Exception as e:
        print(e)

Directory deleted: c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\raw\osm countries queries\Fiji


In [76]:
tsm.donwload_country_data_from_bucket(to_download_countries, bucket_name, task_meta[task]['dir'], ROOT / task_meta[task]['dir'], s3, logger)

[2026-03-24 15:19:00] [INFO] :   * Countries to get data: 1
[2026-03-24 15:19:00] [INFO] :     * Found in B2: 1, Missing in B2: 0
[2026-03-24 15:19:00] [INFO] :   * Downloading data for countries: 1
[2026-03-24 15:19:00] [INFO] :     * Directory: 'data\raw\osm countries queries' -> 'c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\raw\osm countries queries'
[2026-03-24 15:19:00] [INFO] :     * (1/1) Country Fiji files found: 4
[2026-03-24 15:19:00] [INFO] :       * File 'lvl_2_chunk_0_rawOSMRes.json' downloaded successfully to 'lvl_2_chunk_0_rawOSMRes.json'
[2026-03-24 15:19:00] [INFO] :       * File 'lvl_4_chunk_0_rawOSMRes.json' downloaded successfully to 'lvl_4_chunk_0_rawOSMRes.json'
[2026-03-24 15:19:01] [INFO] :       * File 'lvl_6_chunk_0_rawOSMRes.json' downloaded successfully to 'lvl_6_chunk_0_rawOSMRes.json'
[2026-03-24 15:19:01] [INFO] :       * File 'lvl_8_chunk_0_rawOSMRes.json' downloaded successfully to 'lvl_8_chunk_0_rawOSMRes.json'
[2

['Fiji']

In [199]:
countries_missing = ['Canada','Germany','France','Peru']
print(tgm.intersection(sync_state[task]['b2_countries'], countries_missing))
print(tgm.intersection(sync_state[task]['local_countries'], countries_missing))

['France', 'Canada', 'Germany', 'Peru']
['France', 'Canada', 'Germany', 'Peru']


# upload state files

In [ ]:
tsm.upload_file_to_backblaze(DATA_DIR / 'osmMetaCountrDict.json', config)

[2026-01-24 19:12:36] [INFO] : Uploaded c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\osmMetaCountrDict.json to Backblaze successfully


{'status': 'ok', 'status_type': None, 'data': None}

In [41]:
tsm.upload_file_to_backblaze(DATA_DIR / 'process_state.json', config)

[2026-03-23 16:40:37] [INFO] : Uploaded c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\process_state.json to Backblaze successfully


{'status': 'ok', 'status_type': None, 'data': None}

In [95]:
tsm.upload_file_to_backblaze(DATA_DIR / 'first_level_test_state.json', config)

[2026-03-24 17:52:28] [INFO] : Uploaded c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\first_level_test_state.json to Backblaze successfully


{'status': 'ok', 'status_type': None, 'data': None}

In [42]:
tsm.upload_file_to_backblaze(DATA_DIR / 'osmMetaCountrDict.json', config)

[2026-03-23 16:40:53] [INFO] : Uploaded c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\osmMetaCountrDict.json to Backblaze successfully


{'status': 'ok', 'status_type': None, 'data': None}

# upload data

In [162]:
task = 'test_duplicates'
local_countries_not_in_b2 = tgm.complement(sync_state[task]['local_countries'], sync_state[task]['b2_countries'])
len(local_countries_not_in_b2)

28

In [163]:
# countries_to_upload = tgm.intersection(['Canada','Germany','France','Peru'], local_countries_not_in_b2)
countries_to_upload = local_countries_not_in_b2
# countries_to_upload = ['Benin']
print(len(countries_to_upload))

28


In [164]:
dirs = [ROOT / task_meta[task]['dir'] / country for country in countries_to_upload]
print(len(dirs))

28


In [165]:
for dir in dirs:
    tsm.upload_dir_files_to_backblaze(dir, config)

[2026-01-25 16:45:46] [INFO] : Uploading directory: c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\tests results\osm duplicates test\UnitedKingdom
[2026-01-25 16:45:46] [INFO] : Number of files found: 1
[2026-01-25 16:45:46] [INFO] : Uploading UnitedKingdom_dups_test_res_0.pkl ...
[2026-01-25 16:45:47] [INFO] : Uploaded UnitedKingdom_dups_test_res_0.pkl to Backblaze successfully
[2026-01-25 16:45:47] [INFO] : Uploading directory: c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\tests results\osm duplicates test\Bangladesh
[2026-01-25 16:45:47] [INFO] : Number of files found: 1
[2026-01-25 16:45:47] [INFO] : Uploading Bangladesh_dups_test_res_0.pkl ...
[2026-01-25 16:45:48] [INFO] : Uploaded Bangladesh_dups_test_res_0.pkl to Backblaze successfully
[2026-01-25 16:45:48] [INFO] : Uploading directory: c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\tests results\osm duplicates test\S